# 10.7 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/10-deployment-mlops/07-responsible-ai/exercises.ipynb)

In [ ]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
repo_file("data/download.py")                                  # churn_model finds the data through it
sys.path.insert(0, str(repo_file("lib/churn_model.py").parent))
import churn_model as cm

bundle = cm.load_bundle()          # trains and saves models/churn_bundle.joblib the first time
ROOT = cm.repo_root()
print(f"Model trained {bundle['trained']}, test AUC {bundle['metrics']['test_auc']}, "
      f"threshold {bundle['threshold']}")

## Exercise 1: by hand

Group C: 8 customers, flags 1,1,1,1,0,0,0,0, churned 1,1,0,0,1,0,0,0. Group D: 4 customers, flags 1,0,0,0, churned 1,1,0,0. Compute each group's flag rate, recall and false positive rate. Which fairness definition does this model violate more?

In [ ]:
# your code here

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.base import clone
cust = cm.build_customer_table(cm.load_sales())
train, test = train_test_split(cust, test_size=0.25, random_state=42, stratify=cust["churned"])
test = test.assign(p=bundle["pipeline"].predict_proba(test[cm.FEATURES])[:, 1],
                   group=np.where(test["country"] == "United Kingdom", "UK", "non-UK"))
print(test["group"].value_counts().to_string())

def group_metrics(df, threshold):
    """Flag rate, recall (true positive rate) and false positive rate per group."""
    rows = {}
    for name, g in df.groupby("group"):
        flag, churn = g["p"] >= threshold, g["churned"] == 1
        rows[name] = {"customers": len(g), "churners": int(churn.sum()), "flag rate": flag.mean(),
                      "recall": (flag & churn).sum() / churn.sum(),
                      "false positive rate": (flag & ~churn).sum() / (~churn).sum(),
                      "avg predicted": g["p"].mean(), "actual churn": churn.mean()}
    return pd.DataFrame(rows).T

## Exercise 2: Germany vs France

Repeat the group comparison for German and French test customers at threshold 0.35. How many churners does each group have? What can you honestly conclude?

In [ ]:
# your code here

## Exercise 3: equal opportunity by design

Find, for each group (UK, non-UK), the threshold that gives a recall of 0.80 on the training customers' cross-validated predictions. Apply them to the test set. What happens to the flag rates? Would you deploy group-specific thresholds? Discuss.

In [ ]:
# your code here